# Function calling

https://platform.openai.com/docs/guides/function-calling

<img src="https://cdn.openai.com/API/docs/images/function-calling-diagram-steps.png" alt="Function Calling Diagram" width="600"/>

Function Calling은 모델이 직접 외부 API나 Python 함수를 실행하는 기능이 아니다.

모델은 다음 두 가지를 판단한다.

1. 어떤 함수를 호출해야 하는가
2. 함수에 어떤 인자를 전달해야 하는가

실제 함수 실행은 개발자의 코드에서 수행한다.

In [8]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 : ", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 :  gpt-4.1-mini


## 수강생 상담용 함수
- 외부 API 없이 Function Calling 동작 구조를 확인
- 실제 서비스의 경우 DB 조회, 통계 계산, 학습 이력 분석 로직 등으로 바뀔 수 있음

In [4]:
students = {
    "철수" : {"score" : 82, "attendance" : 0.92, "late_count" : 1},
    "관순" : {"score" : 61, "attendance" : 0.76, "late_count" : 4},
    "순신" : {"score" : 95, "attendance" : 0.98, "late_count" : 0}
}

def make_learning_feedback(score, attendance, late_count):
    """점수, 출석률, 지각 횟수를 바탕으로 학습 피드백을 만든다."""
    if score >= 90 and attendance >= 0.9:
        level = "우수"
        message = "현재 흐름이 좋으므로 심화 과제를 제공해도 좋습니다."
    elif score >= 70 and attendance >= 0.85:
        level = "보통"
        message = "기본기는 있으나 오답 유형을 점검하는 보충 학습이 필요합니다."
    else:
        level = "관리 필요"
        message = "출석과 기본 개념 복습을 함께 관리해야 합니다."

    if late_count >= 3:
        message += "지각 횟수가 많으므로 학습 루틴 점검도 필요합니다."

    return {
        "level" : level,
        "feedback" : message
    }

def get_student_counseling(name):
    """학생 이름을 받아 상태 조회와 학습 피드백 생성을 함께 수행한다."""
    data = students.get(name)

    if not data:
        return {
            "found" : False,
            "message" : f"{name} 학생 정보를 찾을 수 없습니다."
        }    
    
    feedback = make_learning_feedback(
        score=data["score"],
        attendance=data["attendance"],
        late_count=data["late_count"],
    )

    return {
        "found" : True,
        "name" : name,
        **data,
        **feedback
    }

print(get_student_counseling('철수'))
print(get_student_counseling('수진'))

{'found': True, 'name': '철수', 'score': 82, 'attendance': 0.92, 'late_count': 1, 'level': '보통', 'feedback': '기본기는 있으나 오답 유형을 점검하는 보충 학습이 필요합니다.'}
{'found': False, 'message': '수진 학생 정보를 찾을 수 없습니다.'}


## Tool Schema 작성
- 모델은 함수를 직접 볼 수 없다. 따라서 함수 이름, 설명, 파라미터 구조를 JSON Schema 형태로 알려줘야 한다.
- 모델은 이 설명을 보고 어떤 상황에서 어떤 함수를 호출할 지 판단한다.

In [5]:
student_tools = [
    {
        "type" : "function",
        "name" : "get_student_counseling",
        "description" : "학생 이름을 받아 점수, 출석률, 지각 횟수와 학습 상담 피드백을 함께 조회한다.",
        "parameters" : {
            "type" : "object",
            "additionalProperties" : False,
            "properties" : {
                "name" : {
                    "type" : "string",
                    "description" : "학생 이름"
                }
            },
            "required" : ["name"]
        }
    }
]

student_function_map = {
    "get_student_counseling" : get_student_counseling
}

## Function Calling 실행 헬퍼

In [10]:
import json

def run_function_call(item, function_map):
    """모델이 요청한 function_call item을 실제 함수 실행 결과로 변환한다."""

    name = item.name                    # 모델이 호출하겠다고 선택한 함수 이름
    args = json.loads(item.arguments)   # 모델이 생성한 함수 인자를 python dict으로 변환
    if name not in function_map:
        result = {
            "error" : f"등록 되지 않은 함수입니다: {name}"
        }
    else:
        result = function_map[name](**args)

    # 실행 결과를 모델에게 다시 전달할 function_call_output 형식으로 변환
    return {
        "type" : "function_call_output",
        "call_id" : item.call_id,
        "output" : json.dumps(result, ensure_ascii=False)
    }

def run_with_tools(prompt, tools, function_map, instructions, max_rounds=5, verbose=True):
    """Function Calling 전체 흐름을 반복 실행한다."""

    input_items = prompt            # 첫 요청에는 사용자 프롬프트를 그대로 전달
    previous_response_id = None     # 이전 응답과 다음 요청을 연결하기 위한 response id

    # 모델이 함수 호출을 여러번 이어갈 수 있으므로 최대 횟수까지 반복
    for round_no in range(1, max_rounds + 1):

        request_args = {
            "model" : DEFAULT_MODEL,
            "instructions" : instructions,
            "input" : input_items,
            "tools" : tools
        }

        if previous_response_id:
            request_args['previous_response_id'] = previous_response_id

        response = client.responses.create(**request_args)

        previous_response_id = response.id

        if verbose:
            print(f'[round {round_no}] output types: ', [item.type for item in response.output])

        function_outputs = []

        # 모델 응답 중 function_call이 있으면 함수를 호출한다
        for item in response.output:
            if item.type == 'function_call':
                function_outputs.append(run_function_call(item, function_map))

        # function_call이 없으면 최종 답변이 생성 된 것이므로 반환
        if not function_outputs:
            return response.output_text
        
        if verbose:
            print(f"[round {round_no}] function outputs:")
            for output in function_outputs:
                print(output)

        # 다음 요청에 실행한 함수 결과를 전달
        input_items = function_outputs 

    return "최대 반복 횟수에 도달했습니다. 함수 호출 흐름을 확인하세요."       

## 수강생 상담 예제 실행

In [8]:
student_instructions = """
너는 수강생 학습 상담을 돕는 AI 보조 강사다.
필요한 경우 제공 된 함수를 사용한다.
학생의 점수, 출석률, 지각 횟수, 상담 피드백 근거를 반영하여 구체적으로 답한다.
"""

answer = run_with_tools(
    prompt="관순 학생의 현재 상태를 보고 상담 코멘트를 작성해줘.",
    tools=student_tools,
    function_map=student_function_map,
    instructions=student_instructions
)

print(answer)

[round 1] output types:  ['function_call']
[round 1] function outputs:
{'type': 'function_call_output', 'call_id': 'call_KZX2kvDgurLA6NGKSVDSar7T', 'output': '{"found": true, "name": "관순", "score": 61, "attendance": 0.76, "late_count": 4, "level": "관리 필요", "feedback": "출석과 기본 개념 복습을 함께 관리해야 합니다.지각 횟수가 많으므로 학습 루틴 점검도 필요합니다."}'}
[round 2] output types:  ['message']
관순 학생은 현재 점수 61점으로 성적이 다소 낮은 편이며 출석률은 76%로 다소 부족한 상태입니다. 지각 횟수가 4회로 잦은 편이어서 학습 루틴 점검이 필요합니다.

상담 코멘트:
관순 학생은 출석률과 지각 횟수를 개선하여 꾸준히 수업에 참여하는 것이 중요합니다. 또한 기본 개념 복습을 병행하여 학습 기초를 다지는 노력이 필요합니다. 지각 습관을 바로잡고 규칙적인 학습 습관을 형성할 수 있도록 도움을 드리겠습니다. 관리가 필요한 단계이므로 꾸준한 관심과 점검을 통해 성적 향상에 힘쓰면 좋겠습니다.


## 외부 API 기반 Function Calling

이번에는 외부 API를 호출하는 함수를 만들고, 모델이 필요한 상황에서 해당 함수를 선택하도록 구성한다.

Open-Meteo는 비상업적 사용 기준으로 API Key 없이 사용할 수 있어 간편하다.

- Geocoding API: https://open-meteo.com/en/docs/geocoding-api
- Forecast API: https://open-meteo.com/en/docs
- Air Quality API: https://open-meteo.com/en/docs/air-quality-api

## 날씨 API

사용자가 도시명을 입력하면 다음 순서로 처리한다.

1. 도시명으로 위도/경도를 조회한다.
2. 위도/경도를 사용해 현재 날씨를 조회한다.
3. 날씨 코드, 기온, 체감 온도, 강수량, 풍속 등을 읽기 쉬운 형태로 정리한다.
4. 모델은 함수 결과를 바탕으로 외출 조언을 작성한다.

모델에게 공개할 함수는 `get_current_weather` 하나이다.

`get_coordinates`는 내부 보조 함수로만 사용한다.

In [11]:
# 외부 API 호출에 사용할 라이브러리
# requests가 설치되어 있지 않다면 아래 명령을 먼저 실행한다.
# %pip install requests -q

import requests

# Open-Meteo API 주소
GEOCODING_API_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_API_URL = "https://api.open-meteo.com/v1/forecast"
AIR_QUALITY_API_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

print("외부 API 호출 준비 완료")

외부 API 호출 준비 완료


In [12]:
# Open-Meteo의 weather_code는 숫자로 내려온다.
# 수업에서 읽기 쉽도록 숫자 코드를 한글 날씨 설명으로 바꾼다.
WEATHER_CODE_MAP = {
    0 : "맑음",
    1 : "대체로 맑음",
    2 : "부분적으로 흐림",
    3 : "흐림",
    45 : "안개",
    48 : "서리 안개",
    51 : "약한 이슬비",
    53 : "이슬비",
    55 : "강한 이슬비",
    61 : "약한 비",
    63 : "비",
    65 : "강한 비",
    71 : "약한 눈",
    73 : "눈",
    75 : "강한 눈",
    80 : "약한 소나기",
    81 : "소나기",
    82 : "강한 소나기",
    95 : "천둥번개",
    96 : "우박을 동반한 천둥번개",
    99 : "강한 우박을 동반한 천둥번개"
}

def get_coordinates(city):
    """도시명을 위도/경도 정보로 변환한다."""

    params = {
        "name" : city,
        "count" : 1,
        "language" : "ko",
        "format" : "json"
    }

    try:
        response = requests.get(GEOCODING_API_URL, params=params, timeout=5)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        return {
            "found" : False,
            "message" : f"도시 정보를 조회하는 중 오류가 발생했습니다: {e}"
        }

    results = data.get("results", [])

    if not results:
        return {
            "found" : False,
            "message" : f"'{city}'에 해당하는 도시를 찾을 수 없습니다."
        }

    location = results[0]

    return {
        "found" : True,
        "name" : location.get("name"),
        "country" : location.get("country"),
        "latitude" : location.get("latitude"),
        "longitude" : location.get("longitude"),
        "timezone" : location.get("timezone")
    }


def get_current_weather(city):
    """도시명을 받아 현재 날씨 정보를 조회한다."""

    location = get_coordinates(city)

    if not location.get("found"):
        return location

    params = {
        "latitude" : location["latitude"],
        "longitude" : location["longitude"],
        # current 파라미터는 현재 시점의 날씨 데이터 중 어떤 항목을 받을지 지정한다.
        # temperature_2m: 지상 2m 높이의 현재 기온
        # relative_humidity_2m: 지상 2m 높이의 상대 습도
        # apparent_temperature: 체감 온도
        # precipitation: 현재 강수량
        # weather_code: 날씨 상태 코드
        # wind_speed_10m: 지상 10m 높이의 풍속
        "current" : "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m",
        "timezone" : location.get("timezone") or "auto"
    }

    try:
        response = requests.get(FORECAST_API_URL, params=params, timeout=5)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        return {
            "found" : False,
            "message" : f"날씨 정보를 조회하는 중 오류가 발생했습니다: {e}"
        }

    current = data.get("current", {})
    weather_code = current.get("weather_code")

    return {
        "found" : True,
        "city" : location["name"],
        "country" : location["country"],
        "time" : current.get("time"),
        "weather" : WEATHER_CODE_MAP.get(weather_code, f"알 수 없는 날씨 코드({weather_code})"),
        "temperature" : current.get("temperature_2m"),
        "apparent_temperature" : current.get("apparent_temperature"),
        "humidity" : current.get("relative_humidity_2m"),
        "precipitation" : current.get("precipitation"),
        "wind_speed" : current.get("wind_speed_10m"),
        "unit" : {
            "temperature" : "°C",
            "precipitation" : "mm",
            "wind_speed" : "km/h",
            "humidity" : "%"
        }
    }

## 날씨 조회 Tool Schema
- 모델에게는 `get_current_weather` 함수의 이름, 설명, 입력 값만 알려준다.
- 실제 API 주소, 요청 파라미터, JSON 파싱 방식은 모델이 알 필요가 없다.

In [13]:
weather_tools = [
    {
        "type" : "function",
        "name" : "get_current_weather",
        "description" : "도시명을 받아 현재 날씨, 기온, 체감 온도, 습도, 강수량, 풍속을 조회한다.",
        "parameters" : {
            "type" : "object",
            "additionalProperties" : False,
            "properties" : {
                "city" : {
                    "type" : "string",
                    "description" : "날씨를 조회할 도시명. Open-Meteo 검색 정확도를 높히기 위해 영문 도시명으로 입력한다. 예: 서울 -> Seoul, 부산 -> Busan, 도쿄 -> Tokyo, 뉴욕 -> New York"
                }
            },
            "required" : ["city"]
        }
    }
]

weather_function_map = {
    "get_current_weather" : get_current_weather
}

## 날씨 API Function Calling 실행

In [14]:
weather_instructions = """
너는 날씨 정보를 알려주는 assistant이다.

사용자가 특정 도시의 날짜를 물어보면 get_current_weather 함수를 호출한다.

함수 호출 시 city 인자는 Open-Meteo 검색 정확도를 높이기 위해 영문 도시명으로 변환해서 전달한다.
예: 서울 -> Seoul, 부산 -> Busan, 도쿄 -> Tokyo, 뉴욕 -> New York

함수 호출 결과를 바탕으로 현재 날씨, 기온, 체감 온도, 습도, 강수량, 풍속을 자연스럽게 설명한다.
비나 눈이 오는 경우 우산 또는 외출 주의 여부를 함께 안내한다.

날씨 데이터가 조회되지 않으면 임의로 추측하지 말고 조회에 실패했다고 안내한다.
"""

answer = run_with_tools(
    prompt="제주도 오늘 놀러갈껀데 날씨 확인해서 옷차림은 어떻게 하면 좋을지 알려줘.",
    tools=weather_tools,
    function_map=weather_function_map,
    instructions=weather_instructions
)

answer

[round 1] output types:  ['function_call']
[round 1] function outputs:
{'type': 'function_call_output', 'call_id': 'call_4wkkKgP6sI36xqcI3248hvRR', 'output': '{"found": true, "city": "Jeju", "country": "에티오피아", "time": "2026-05-08T03:45", "weather": "대체로 맑음", "temperature": 19.1, "apparent_temperature": 17.8, "humidity": 56, "precipitation": 0.0, "wind_speed": 9.6, "unit": {"temperature": "°C", "precipitation": "mm", "wind_speed": "km/h", "humidity": "%"}}'}
[round 2] output types:  ['message']


'제주도의 오늘 날씨는 대체로 맑음이며, 기온은 약 19.1도, 체감 온도는 약 17.8도입니다. 습도는 56%, 강수량은 없고, 바람은 시속 9.6km로 조금 부는 편입니다.\n\n옷차림은 약간 선선할 수 있으니 가벼운 긴팔이나 얇은 자켓을 입는 것이 좋습니다. 맑은 날씨이므로 야외 활동하기에 적합하며, 우산은 필요 없을 것 같습니다. 즐거운 여행 되세요!'

## 실습: 미세먼지 API와 날씨 API 조합하기

사용자가 "오늘 서울에서 러닝해도 될까?"처럼 질문했을 때, 날씨와 대기질을 함께 보고 판단하는 것이 목표이다.

Open-Meteo Air Quality API에서 사용할 주요 값은 다음과 같다.

| 항목 | 의미 |
|---|---|
| `pm10` | 미세먼지 농도 |
| `pm2_5` | 초미세먼지 농도 |
| `european_aqi` | 유럽 기준 대기질 지수 |
| `us_aqi` | 미국 기준 대기질 지수 |
| `ozone` | 오존 |

In [15]:
def get_pm25_grade(pm25):
    """초미세먼지 값을 간단한 등급으로 변환한다."""

    if pm25 is None:
        return "정보 없음"
    if pm25 <= 15:
        return "좋음"
    if pm25 <= 35:
        return "보통"
    if pm25 <= 75:
        return "나쁨"
    return "매우 나쁨"


def get_pm10_grade(pm10):
    """미세먼지 값을 간단한 등급으로 변환한다."""

    if pm10 is None:
        return "정보 없음"
    if pm10 <= 30:
        return "좋음"
    if pm10 <= 80:
        return "보통"
    if pm10 <= 150:
        return "나쁨"
    return "매우 나쁨"


def get_air_quality(city):
    """도시명을 받아 현재 미세먼지와 대기질 정보를 조회한다."""

    location = get_coordinates(city)

    if not location.get("found"):
        return location

    params = {
        "latitude" : location["latitude"],
        "longitude" : location["longitude"],
        # current 파라미터는 현재 시점의 대기질 데이터 중 어떤 항목을 받을지 지정한다.
        # european_aqi: 유럽 기준 대기질 지수
        # us_aqi: 미국 기준 대기질 지수
        # pm10: 미세먼지 농도, 지름 10마이크로미터 이하 입자
        # pm2_5: 초미세먼지 농도, 지름 2.5마이크로미터 이하 입자
        # carbon_monoxide: 일산화탄소 농도
        # nitrogen_dioxide: 이산화질소 농도
        # sulphur_dioxide: 이산화황 농도
        # ozone: 오존 농도
        "current" : "european_aqi,us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone",
        "timezone" : location.get("timezone") or "auto"
    }

    try:
        response = requests.get(AIR_QUALITY_API_URL, params=params, timeout=5)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        return {
            "found" : False,
            "message" : f"대기질 정보를 조회하는 중 오류가 발생했습니다: {e}"
        }

    current = data.get("current", {})
    pm10 = current.get("pm10")
    pm25 = current.get("pm2_5")

    return {
        "found" : True,
        "city" : location["name"],
        "country" : location["country"],
        "time" : current.get("time"),
        "european_aqi" : current.get("european_aqi"),
        "us_aqi" : current.get("us_aqi"),
        "pm10" : pm10,
        "pm2_5" : pm25,
        "pm10_grade" : get_pm10_grade(pm10),
        "pm2_5_grade" : get_pm25_grade(pm25),
        "carbon_monoxide" : current.get("carbon_monoxide"),
        "nitrogen_dioxide" : current.get("nitrogen_dioxide"),
        "sulphur_dioxide" : current.get("sulphur_dioxide"),
        "ozone" : current.get("ozone"),
        "unit" : {
            "pm10" : "μg/m³",
            "pm2_5" : "μg/m³",
            "carbon_monoxide" : "μg/m³",
            "nitrogen_dioxide" : "μg/m³",
            "sulphur_dioxide" : "μg/m³",
            "ozone" : "μg/m³"
        }
    }


def get_outdoor_activity_context(city, activity):
    """날씨와 대기질을 함께 조회하여 야외 활동 판단에 필요한 정보를 만든다."""

    weather = get_current_weather(city)
    air_quality = get_air_quality(city)

    if not weather.get("found"):
        return weather

    if not air_quality.get("found"):
        return air_quality

    return {
        "found" : True,
        "city" : weather["city"],
        "country" : weather["country"],
        "activity" : activity,
        "weather" : weather,
        "air_quality" : air_quality
    }

## 미세먼지 조합 Tool Schema 작성

In [16]:
outdoor_tools = [
    {
        "type" : "function",
        "name" : "get_outdoor_activity_context",
        "description" : "도시명과 야외 활동 종류를 받아 현재 날씨와 대기질 정보를 조회한다.",
        "parameters" : {
            "type" : "object",
            "additionalProperties" : False,
            "properties" : {
                "city" : {
                    "type" : "string",
                    "description" : "날씨를 조회할 도시명. Open-Meteo 검색 정확도를 높히기 위해 영문 도시명으로 입력한다. 예: 서울 -> Seoul, 부산 -> Busan, 도쿄 -> Tokyo, 뉴욕 -> New York"
                },
                "activity" :{
                    "type" : "string",
                    "description" : "사용자가 하려는 야외 활동. 예: 산책, 러닝, 등산, 자전거"
                }
            },
            "required" : ["city","activity"]
        }
    }
]

outdoor_function_map = {
    "get_outdoor_activity_context" : get_outdoor_activity_context
}

## 미세먼지 조합 Function Calling 실행

모델은 함수 결과를 바탕으로 다음을 판단한다.

1. 비나 강수량 때문에 활동이 어려운가
2. 바람이 너무 강하지 않은가
3. 초미세먼지나 미세먼지가 나쁜 편인가
4. 사용자가 하려는 활동에 맞는 주의사항은 무엇인가

이처럼 Function Calling은 "데이터 조회"와 "데이터 해석"을 분리해서 설계할 때 효과적이다.

In [17]:
outdoor_instructions = """
너는 날씨와 대기질 정보를 바탕으로 야외 활동 가능 여부를 판단하는 assistant다.

사용자가 특정 도시의 야외 활동 가능 여부를 물어보면 get_outdoor_activity_context 함수를 호출한다.

함수 호출 시 city 인자는 Open-Meteo 검색 정확도를 높이기 위해 가능한 영문 도시명으로 변환해서 전달한다.
예: 서울 -> Seoul, 부산 -> Busan, 도쿄 -> Tokyo, 뉴욕 -> New York

날씨와 대기질 정보는 반드시 함수 실행 결과를 기준으로 답한다.
사용자가 하려는 활동을 고려하여 가능 여부, 주의사항, 대체 제안을 구체적으로 안내한다.
특히 강수량, 풍속, 기온, 미세먼지(pm10), 초미세먼지(pm2_5)를 함께 반영한다.
API 결과에 없는 내용은 확정적으로 말하지 않는다.
"""

answer = run_with_tools(
    prompt="오늘 서울에서 1시간 정도 러닝해도 괜찮을까?",
    tools=outdoor_tools,
    function_map=outdoor_function_map,
    instructions=outdoor_instructions
)

answer

[round 1] output types:  ['function_call']
[round 1] function outputs:
{'type': 'function_call_output', 'call_id': 'call_cyeYzjWvkgIORWqeJyJ4Cshr', 'output': '{"found": true, "city": "서울특별시", "country": "대한민국", "activity": "러닝", "weather": {"found": true, "city": "서울특별시", "country": "대한민국", "time": "2026-05-08T10:30", "weather": "맑음", "temperature": 16.2, "apparent_temperature": 15.3, "humidity": 51, "precipitation": 0.0, "wind_speed": 9.6, "unit": {"temperature": "°C", "precipitation": "mm", "wind_speed": "km/h", "humidity": "%"}}, "air_quality": {"found": true, "city": "서울특별시", "country": "대한민국", "time": "2026-05-08T10:00", "european_aqi": 42, "us_aqi": 68, "pm10": 20.7, "pm2_5": 11.9, "pm10_grade": "좋음", "pm2_5_grade": "좋음", "carbon_monoxide": 272.0, "nitrogen_dioxide": 13.5, "sulphur_dioxide": 17.1, "ozone": 72.0, "unit": {"pm10": "μg/m³", "pm2_5": "μg/m³", "carbon_monoxide": "μg/m³", "nitrogen_dioxide": "μg/m³", "sulphur_dioxide": "μg/m³", "ozone": "μg/m³"}}}'}
[round 2] output ty

'오늘 서울에서 1시간 정도 러닝하기에 적합한 날씨입니다. 현재 서울의 날씨는 맑고 기온은 16.2도이며 강수량은 0.0mm로 비가 내리지 않고 있습니다. 바람도 9.6km/h로 큰 불편 없이 러닝할 수 있는 수준입니다.\n\n대기질도 매우 좋음 등급으로 PM10은 20.7μg/m³, PM2.5는 11.9μg/m³로 건강한 공기 상태입니다.\n\n따라서 오늘 러닝을 하셔도 무방하지만, 평소보다 약간 선선하니 운동복을 적절히 선택하시고 물을 충분히 섭취하시면서 진행하시면 좋겠습니다. 즐거운 러닝 되세요!'